Import Modules

In [10]:
import math
import torch
import torch.nn as nn
from transformers import AutoTokenizer

### `TokenEmbedding`

**Accepts:**

* `vocab_size` : Number of unique tokens in the vocabulary.
* `embedding_dim` : Size of each token embedding vector.
* `x` : Input token IDs.

**Does:**

* Converts token IDs into embedding vectors using `nn.Embedding`.
* Scales the embeddings by `√embedding_dim` as done in the original Transformer paper.

**Returns:**

* Scaled token embeddings.


In [11]:
class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.embedding_dim = embedding_dim

    def forward(self, x):
        # Scale embeddings as in the original Transformer paper
        return self.embedding(x) * math.sqrt(self.embedding_dim)

### `PositionalEncoding`

**Accepts:**

* `embedding_dim` : Size of each token embedding vector.
* `max_length` : Maximum sequence length.
* `x` : Token embeddings.

**Does:**

* Creates sinusoidal position vectors using sine and cosine.
* Adds position information to token embeddings.
* Stores positional encoding as a non-trainable buffer.

**Returns:**

* `x` : Embeddings with positional information added.


In [12]:
class PositionalEncoding(nn.Module):
    def __init__(self, embedding_dim, max_length=5000):
        super().__init__()

        position = torch.arange(max_length).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, embedding_dim, 2) *
            (-math.log(10000.0) / embedding_dim)
        )

        pe = torch.zeros(max_length, embedding_dim)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        seq_len = x.size(1)
        return x + self.pe[:, :seq_len]

### `ScaledDotProductAttention`

**Accepts:**

* `Q` : Query matrix
* `K` : Key matrix
* `V` : Value matrix
* `mask` *(optional)* : Attention mask

**Does:**

* Computes `Q × Kᵀ`
* Scales by `√d_k`
* Applies mask (if provided)
* Applies Softmax
* Multiplies attention weights with `V`

**Returns:**

* `output` : Contextual embeddings
* `attention_weights` : Attention probabilities


In [13]:
class ScaledDotProductAttention(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, Q, K, V, mask=None):

        d_k = Q.size(-1)

        scores = torch.matmul(Q, K.transpose(-2, -1))
        scores = scores / math.sqrt(d_k)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)

        attention_weights = torch.softmax(scores, dim=-1)

        output = torch.matmul(attention_weights, V)

        return output, attention_weights

### `MultiHeadSelfAttention`

**Accepts:**

* `embedding_dim` : Size of each embedding vector.
* `num_heads` : Number of attention heads.
* `x` : Input embeddings.
* `mask` *(optional)* : Attention mask.

**Does:**

* Creates **Q, K, V** using three `nn.Linear` layers.
* Splits Q, K, and V into multiple heads.
* Applies **Scaled Dot-Product Attention** to each head.
* Concatenates all head outputs.
* Applies a final `nn.Linear` layer.

**Returns:**

* `output` : Multi-head contextual embeddings.
* `attention_weights` : Attention probabilities from all heads.



In [14]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, embedding_dim, num_heads):
        super().__init__()

        assert embedding_dim % num_heads == 0

        self.embedding_dim = embedding_dim
        self.num_heads = num_heads
        self.head_dim = embedding_dim // num_heads

        self.Wq = nn.Linear(embedding_dim, embedding_dim)
        self.Wk = nn.Linear(embedding_dim, embedding_dim)
        self.Wv = nn.Linear(embedding_dim, embedding_dim)

        self.output = nn.Linear(embedding_dim, embedding_dim)

        self.attention = ScaledDotProductAttention()

    def forward(self, x, mask=None):

        batch_size, seq_len, _ = x.shape

        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)

        Q = Q.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        K = K.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        V = V.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        attention_output, attention_weights = self.attention(
            Q,
            K,
            V,
            mask
        )

        attention_output = attention_output.transpose(1, 2)

        attention_output = attention_output.contiguous().view(
            batch_size,
            seq_len,
            self.embedding_dim
        )

        output = self.output(attention_output)

        return output, attention_weights

### `FeedForwardNetwork`

**Accepts:**

* `embedding_dim` : Size of the input embedding.
* `hidden_dim` : Size of the hidden layer.
* `x` : Contextual embeddings from the attention layer.

**Does:**

* Applies a first `nn.Linear` to expand the embedding.
* Applies `ReLU` activation.
* Applies a second `nn.Linear` to project back to the original embedding size.

**Returns:**

* Improved contextual embeddings ready for the next layer.


In [15]:
class FeedForwardNetwork(nn.Module):
    def __init__(self, embedding_dim, hidden_dim):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, embedding_dim),
        )

    def forward(self, x):
        return self.network(x)

### `EncoderBlock`

**Accepts:**

* `embedding_dim` : Size of each embedding vector.
* `num_heads` : Number of attention heads.
* `hidden_dim` : Hidden size of the FFN.
* `dropout` : Dropout rate.
* `x` : Input embeddings.
* `mask` *(optional)* : Attention mask.

**Does:**

* Applies **Multi-Head Self-Attention**.
* Applies **Add & Norm** (Residual + LayerNorm).
* Applies **Feed-Forward Network (FFN)**.
* Applies another **Add & Norm**.

**Returns:**

* `x` : Updated contextual embeddings.
* `attention_weights` : Attention probabilities.


In [16]:
class EncoderBlock(nn.Module):
    def __init__(
        self,
        embedding_dim,
        num_heads,
        hidden_dim,
        dropout=0.1,
    ):
        super().__init__()

        self.attention = MultiHeadSelfAttention(
            embedding_dim,
            num_heads
        )

        self.ffn = FeedForwardNetwork(
            embedding_dim,
            hidden_dim
        )

        self.norm1 = nn.LayerNorm(embedding_dim)
        self.norm2 = nn.LayerNorm(embedding_dim)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):

        attention_output, attention_weights = self.attention(
            x,
            mask
        )

        x = self.norm1(
            x + self.dropout(attention_output)
        )

        ffn_output = self.ffn(x)

        x = self.norm2(
            x + self.dropout(ffn_output)
        )

        return x, attention_weights

### `TransformerEncoder`

**Accepts:**

* `vocab_size` : Number of unique tokens.
* `embedding_dim` : Size of each embedding vector.
* `num_heads` : Number of attention heads.
* `hidden_dim` : Hidden size of the FFN.
* `num_layers` : Number of Encoder Blocks.
* `num_classes` : Number of output classes.
* `dropout` : Dropout rate.
* `x` : Input token IDs.
* `mask` *(optional)* : Attention mask.

**Does:**

* Converts token IDs into embeddings.
* Adds positional encoding.
* Applies embedding dropout.
* Passes embeddings through all Encoder Blocks.
* Takes the **`[CLS]` token** representation.
* Uses a `nn.Linear` classification head to produce class scores.

**Returns:**

* `logits` : Raw class scores (before Softmax).
* `attention_scores` : Attention weights from each Encoder Block.


In [17]:
class TransformerEncoder(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim,
        num_heads,
        hidden_dim,
        num_layers,
        num_classes,
        dropout=0.1,
    ):
        super().__init__()

        self.embedding = TokenEmbedding(
            vocab_size,
            embedding_dim
        )

        self.position = PositionalEncoding(
            embedding_dim
        )

        # Dropout after adding positional encoding
        self.embedding_dropout = nn.Dropout(dropout)

        self.layers = nn.ModuleList(
            [
                EncoderBlock(
                    embedding_dim,
                    num_heads,
                    hidden_dim,
                    dropout
                )
                for _ in range(num_layers)
            ]
        )

        # Classification head
        self.classifier = nn.Linear(
            embedding_dim,
            num_classes
        )


    def forward(self, x, mask=None):

        # Token embedding
        x = self.embedding(x)

        # Add positional information
        x = self.position(x)

        # Apply embedding dropout
        x = self.embedding_dropout(x)

        attention_scores = []

        # Pass through encoder blocks
        for layer in self.layers:
            x, weights = layer(
                x,
                mask
            )

            attention_scores.append(weights)


        # Use [CLS] token representation
        sentence_vector = x[:, 0, :]


        # Convert vector into class scores
        logits = self.classifier(
            sentence_vector
        )

        return logits, attention_scores

### Test / Inference Code

**Accepts:**

* `text` : Input sentence.
* `tokenizer` : Converts text into token IDs.
* `input_ids` : Token IDs given to the Transformer.
* `padding_mask` : Mask that ignores padding tokens.
* Model parameters:

  * `vocab_size` : Vocabulary size.
  * `embedding_dim` : Embedding size.
  * `num_heads` : Number of attention heads.
  * `hidden_dim` : FFN hidden size.
  * `num_layers` : Number of Encoder Blocks.
  * `num_classes` : Number of output classes.

---

**Does:**

* Loads the BERT tokenizer.
* Converts text into token IDs.
* Creates a padding mask.
* Creates the Transformer Encoder model.
* Passes tokens through the encoder.
* Gets classification logits.
* Converts logits into probabilities using Softmax.
* Chooses the class with the highest probability.

---

**Returns:**

* `logits` : Raw prediction scores from the classifier.
* `probabilities` : Class probabilities.
* `prediction` : Final predicted class. (Example: class 0 or class 1)

---

**Full flow:**

```text
Text
 ↓
Tokenizer
 ↓
Token IDs
 ↓
Transformer Encoder
 ↓
[CLS] Vector
 ↓
Linear Classifier
 ↓
Logits
 ↓
Softmax
 ↓
Prediction
```


In [18]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

text = "my exam was bad."

tokens = tokenizer(
    text,
    padding=True,
    return_tensors="pt"
)

input_ids = tokens["input_ids"]

# Create padding mask
# 1 = allowed to attend
# 0 = ignore

padding_mask = (
    input_ids != tokenizer.pad_token_id
)

# Attention expects:
# [batch, 1, 1, sequence_length]

padding_mask = padding_mask.unsqueeze(1).unsqueeze(2)

vocab_size = tokenizer.vocab_size

model = TransformerEncoder(
    vocab_size=vocab_size,
    embedding_dim=512,
    num_heads=8,
    hidden_dim=2048,
    num_layers=6,
    num_classes=2,
)

logits, attention = model(
    input_ids,
    padding_mask
)

print("Logits shape:", logits.shape)

print("Logits:", logits
)

# Convert logits to probabilities
probabilities = torch.softmax(
    logits,
    dim=-1
)

print(
    "Probabilities:",
    probabilities
)

# Get predicted class

prediction = torch.argmax(
    probabilities,
    dim=-1
)

print(
    "Predicted class:",
    prediction.item()
)

Logits shape: torch.Size([1, 2])
Logits: tensor([[ 0.0877, -0.5317]], grad_fn=<AddmmBackward0>)
Probabilities: tensor([[0.6501, 0.3499]], grad_fn=<SoftmaxBackward0>)
Predicted class: 0
